# **FRAMEWORK V7: NOTEBOOK DE EXTRACCIÓN DE DATOS CRUDOS**

CAPA - HIDRAULICA DE MASAS

## **M0. Configuración General**

Ingesta de Datos Manual: https://www.datos.gov.co/dataset/Reservas-Hidr-ulicas-en-Masa/5cbd-e6b5/about_data

## **M1. Definición de la Fuente Oficial**

Se define la fuente oficial para la ingesta de datos, que en este caso es el archivo CSV disponible en GitHub. El proceso de carga se realiza mediante la librería `pandas`.

La incorporación de la serie de Reservas Hidráulicas representa un salto cualitativo en la capacidad predictiva del modelo. La data cruda, proveniente de la serie histórica de embalses (XM), se caracteriza por ser de alta frecuencia (diaria), lo cual requiere una transformación rigurosa para alinearse con la escala mensual del modelo maestro.

La variable central de interés es el VolumenUtilDiarioMasa específico para el CodigoEmbalse == 'AGREGADO_BOGOTA', ya que este indicador actúa como un proxy del estado de almacenamiento del sistema hídrico que influye directamente en las variables de calidad (turbidez, conductividad y oxígeno disuelto). La transformación implica la consolidación estadística de los volúmenes para obtener promedios mensuales, garantizando así la coherencia temporal con el Dataset.

## **M2. Extracción y Procesamiento de Datos**

Esta sección se encarga de extraer la información relevante y transformarla según los requerimientos del Framework V7. Se aplica el filtro por `CodigoEmbalse == "AGREGADO_BOGOTA"`, se seleccionan las variables de interés (`Fecha`, `VolumenUtilDiarioMasa`, `VertimientosMasa`), se convierte la columna `Fecha` a tipo datetime y finalmente se realiza un remuestreo mensual para calcular el promedio de las variables numéricas.

In [ ]:
import pandas as pd
import requests
import io

print("="*70)
print("CARGA DE LA CAPA HIDRÁULICA - RESERVAS HIDRÁULICAS")
print("="*70)

#--------------------------------------------------------
# Archivo desde GitHub
#--------------------------------------------------------

url = "https://raw.githubusercontent.com/jriatiga/dataset/refs/heads/main/proyectoGradoV4/05_Capa_Hidraulica/01_Capa_Reservas_Hidraulicas_Masa_V1.csv"

response = requests.get(url)

df = pd.read_csv(io.BytesIO(response.content))

print("\nArchivo cargado correctamente.\n")

#--------------------------------------------------------
# Información general
#--------------------------------------------------------

print("="*70)
print("DIMENSIONES")
print("="*70)

print(df.shape)

print()

print("="*70)
print("COLUMNAS")
print("="*70)

for c in df.columns:
    print(c)

print()

print("="*70)
print("TIPOS DE DATOS")
print("="*70)

print(df.dtypes)

print()

print("="*70)
print("PRIMEROS REGISTROS")
print("="*70)

print(df.head())

print()

print("="*70)
print("ÚLTIMOS REGISTROS")
print("="*70)

print(df.tail())

print()

print("="*70)
print("VALORES NULOS")
print("="*70)

print(df.isnull().sum())

print()

print("="*70)
print("REGISTROS DUPLICADOS")
print("="*70)

print(df.duplicated().sum())

print()

print("="*70)
print("MEMORIA")
print("="*70)

print(df.memory_usage(deep=True))

print()

print("="*70)
print("ESTADÍSTICAS")
print("="*70)

print(df.describe(include="all"))

#--------------------------------------------------------
# Exploración de variables
#--------------------------------------------------------

print()

print("="*70)
print("VARIABLES CATEGÓRICAS")
print("="*70)

for c in df.select_dtypes(include="object").columns:

    print()

    print(c)

    print("-"*40)

    print(df[c].value_counts().head(20))

#--------------------------------------------------------
# Variables numéricas
#--------------------------------------------------------

print()

print("="*70)
print("VARIABLES NUMÉRICAS")
print("="*70)

print(df.select_dtypes(include="number").describe())

#--------------------------------------------------------
# Fechas
#--------------------------------------------------------

if "Fecha" in df.columns:

    df["Fecha"] = pd.to_datetime(df["Fecha"])

    print()

    print("="*70)
    print("COBERTURA TEMPORAL")
    print("="*70)

    print("Inicio :", df["Fecha"].min())

    print("Fin    :", df["Fecha"].max())

#--------------------------------------------------------
# Embalses
#--------------------------------------------------------

if "CodigoEmbalse" in df.columns:

    print()

    print("="*70)
    print("EMBALSES DISPONIBLES")
    print("="*70)

    print(df["CodigoEmbalse"].value_counts())

#--------------------------------------------------------
# Variables hidráulicas
#--------------------------------------------------------

print()

print("="*70)
print("COLUMNAS DEL DATASET")
print("="*70)

print(list(df.columns))

print()

print("="*70)
print("FIN DE LA AUDITORÍA")
print("="*70)

CARGA DE LA CAPA HIDRÁULICA - RESERVAS HIDRÁULICAS

Archivo cargado correctamente.

DIMENSIONES
(63991, 8)

COLUMNAS
FechaPublicacion
Fecha
CodigoEmbalse
RegionHidrologica
CapacidadUtilMasa
VolumenUtilDiarioMasa
VolumenTotalMasa
VertimientosMasa

TIPOS DE DATOS
FechaPublicacion         object
Fecha                    object
CodigoEmbalse            object
RegionHidrologica        object
CapacidadUtilMasa         int64
VolumenUtilDiarioMasa     int64
VolumenTotalMasa          int64
VertimientosMasa          int64
dtype: object

PRIMEROS REGISTROS
  FechaPublicacion       Fecha    CodigoEmbalse RegionHidrologica  \
0       2023-12-19  2019-10-13     AGREGADO_SIN          Colombia   
1       2023-12-19  2019-10-13           PLAYAS         Antioquia   
2       2023-12-19  2019-10-13           PORCE3         Antioquia   
3       2023-12-19  2019-10-13  AGREGADO_BOGOTA            Centro   
4       2023-12-19  2019-10-13            MIEL1            Caldas   

   CapacidadUtilMasa  VolumenUtil

In [ ]:
#--------------------------------------------------------
# Procesamiento de Datos
#--------------------------------------------------------

print("="*70)
print("INICIANDO PROCESAMIENTO DE DATOS")
print("="*70)

# 1. Filtrar por CodigoEmbalse == "AGREGADO_BOGOTA"
df_filtered = df[df['CodigoEmbalse'] == 'AGREGADO_BOGOTA'].copy()
print(f"Número de registros después de filtrar por 'AGREGADO_BOGOTA': {len(df_filtered)}")

# 2. Seleccionar columnas de interés
df_selected = df_filtered[['Fecha', 'VolumenUtilDiarioMasa', 'VertimientosMasa']]
print("Columnas seleccionadas: 'Fecha', 'VolumenUtilDiarioMasa', 'VertimientosMasa'")

# 3. Convertir 'Fecha' a tipo DateTime
df_selected['Fecha'] = pd.to_datetime(df_selected['Fecha'])
print("Columna 'Fecha' convertida a tipo datetime.")

# 4. Resample y Agreggate (Promedio Mensual)
# Establecer 'Fecha' como índice para el remuestreo
df_resampled = df_selected.set_index('Fecha')

# Remuestrear a frecuencia mensual ('MS' - Month Start) y calcular el promedio
df_monthly_avg = df_resampled.resample('MS').mean().reset_index()

print("Datos remuestreados a frecuencia mensual y promediados.")

print("\nPrimeros registros del DataFrame procesado (promedio mensual):")
display(df_monthly_avg.head())

print("\nÚltimos registros del DataFrame procesado (promedio mensual):")
display(df_monthly_avg.tail())

print(f"\nDimensiones del DataFrame procesado: {df_monthly_avg.shape}")
print(f"Tipos de datos del DataFrame procesado:\n{df_monthly_avg.dtypes}")

INICIANDO PROCESAMIENTO DE DATOS
Número de registros después de filtrar por 'AGREGADO_BOGOTA': 2200
Columnas seleccionadas: 'Fecha', 'VolumenUtilDiarioMasa', 'VertimientosMasa'
Columna 'Fecha' convertida a tipo datetime.
Datos remuestreados a frecuencia mensual y promediados.

Primeros registros del DataFrame procesado (promedio mensual):


/tmp/ipykernel_1008/209948463.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Fecha'] = pd.to_datetime(df_selected['Fecha'])


,Fecha,VolumenUtilDiarioMasa,VertimientosMasa
0,2019-01-01,3.753109e+08,0.0
1,2019-02-01,3.492289e+08,0.0
2,2019-03-01,3.222146e+08,0.0
3,2019-04-01,3.082495e+08,0.0
4,2019-05-01,3.076062e+08,0.0



Últimos registros del DataFrame procesado (promedio mensual):


,Fecha,VolumenUtilDiarioMasa,VertimientosMasa
68,2024-09-01,4.868488e+08,0.0
69,2024-10-01,4.701902e+08,0.0
70,2024-11-01,4.816374e+08,0.0
71,2024-12-01,4.923464e+08,0.0
72,2025-01-01,4.784729e+08,0.0



Dimensiones del DataFrame procesado: (73, 3)
Tipos de datos del DataFrame procesado:
Fecha                    datetime64[ns]
VolumenUtilDiarioMasa           float64
VertimientosMasa                float64
dtype: object


## **M3. Reporte de Auditoría**

Se genera un reporte de auditoría detallado tanto para el dataframe original como para el dataframe procesado (`df_monthly_avg`). Este reporte incluye información sobre dimensiones, tipos de datos, valores nulos, estadísticas descriptivas y una muestra de los primeros y últimos registros. Este reporte se exporta a un archivo `Auditoria_Capa_Hidraulica.xlsx`.

In [ ]:
import io
from contextlib import redirect_stdout

print("="*70)
print("GENERANDO REPORTE DE AUDITORÍA")
print("="*70)

# Create a string buffer to capture print outputs for a verbose log file
f = io.StringIO()

with redirect_stdout(f):
    print("--- REPORTE DE AUDITORÍA DEL DATAFRAME ORIGINAL ---")
    print("\nDIMENSIONES:")
    print(df.shape)
    print("\nTIPOS DE DATOS:")
    print(df.dtypes)
    print("\nVALORES NULOS:")
    print(df.isnull().sum())
    print("\nREGISTROS DUPLICADOS:")
    print(df.duplicated().sum())
    print("\nESTADÍSTICAS DESCRIPTIVAS:")
    print(df.describe(include="all"))
    print("\nVARIABLES CATEGÓRICAS (Value Counts):")
    for c in df.select_dtypes(include="object").columns:
        print(f"\n--- {c} ---")
        print(df[c].value_counts().head(20))
    print("\nPRIMEROS REGISTROS:")
    print(df.head())
    print("\nÚLTIMOS REGISTROS:")
    print(df.tail())

    print("\n\n--- REPORTE DE AUDITORÍA DEL DATAFRAME PROCESADO (df_monthly_avg) ---")
    print("\nDIMENSIONES:")
    print(df_monthly_avg.shape)
    print("\nTIPOS DE DATOS:")
    print(df_monthly_avg.dtypes)
    print("\nVALORES NULOS:")
    print(df_monthly_avg.isnull().sum())
    print("\nESTADÍSTICAS DESCRIPTIVAS:")
    print(df_monthly_avg.describe())
    print("\nPRIMEROS REGISTROS:")
    print(df_monthly_avg.head())
    print("\nÚLTIMOS REGISTROS:")
    print(df_monthly_avg.tail())

audit_report_text = f.getvalue()

# Save the captured output to a text file for a verbose audit log
with open('Auditoria_Capa_Hidraulica_Report.txt', 'w') as file:
    file.write(audit_report_text)

print("\nReporte de auditoría generado en 'Auditoria_Capa_Hidraulica_Report.txt'.")

# Export structured audit information to an Excel file with multiple sheets
with pd.ExcelWriter('Auditoria_Capa_Hidraulica.xlsx') as writer:
    # Original Data Audit Sheets
    df.head().to_excel(writer, sheet_name='Raw_Data_Head', index=False)
    df.tail().to_excel(writer, sheet_name='Raw_Data_Tail', index=False)
    df.dtypes.to_frame(name='DataType').to_excel(writer, sheet_name='Raw_Data_Types')
    df.isnull().sum().to_frame(name='NullCount').to_excel(writer, sheet_name='Raw_Data_Nulls')
    df.describe(include='all').to_excel(writer, sheet_name='Raw_Data_Describe')

    # Processed Data Audit Sheets
    df_monthly_avg.head().to_excel(writer, sheet_name='Processed_Data_Head', index=False)
    df_monthly_avg.tail().to_excel(writer, sheet_name='Processed_Data_Tail', index=False)
    df_monthly_avg.dtypes.to_frame(name='DataType').to_excel(writer, sheet_name='Processed_Data_Types')
    df_monthly_avg.isnull().sum().to_frame(name='NullCount').to_excel(writer, sheet_name='Processed_Data_Nulls')
    df_monthly_avg.describe().to_excel(writer, sheet_name='Processed_Data_Describe')

print("Reporte de auditoría estructurado en 'Auditoria_Capa_Hidraulica.xlsx'.")

GENERANDO REPORTE DE AUDITORÍA

Reporte de auditoría generado en 'Auditoria_Capa_Hidraulica_Report.txt'.
Reporte de auditoría estructurado en 'Auditoria_Capa_Hidraulica.xlsx'.


## **M4. Exportación de Resultados**

El dataframe procesado, `df_monthly_avg`, que contiene las reservas hidráulicas promediadas mensualmente para `AGREGADO_BOGOTA`, se exporta a un archivo Excel (`01_Capa_Hydraulica_V1.xlsx`) para su uso posterior en el Framework V7.

In [ ]:
#--------------------------------------------------------
# Exportación de Resultados
#--------------------------------------------------------

print("="*70)
print("EXPORTANDO RESULTADOS")
print("="*70)

output_filename = '01_Capa_Hydraulica_V1.xlsx'
df_monthly_avg.to_excel(output_filename, index=False)
print(f"DataFrame procesado exportado a '{output_filename}'.")

EXPORTANDO RESULTADOS
DataFrame procesado exportado a '01_Capa_Hydraulica_V1.xlsx'.


## **M5. Resumen Final**

Este documento resume las conclusiones del análisis científico de la capa de reservas hidráulicas y define la estrategia de transformación para su integración en el **Framework V7**.

---

**1. Diagnóstico Científico de la Capa**

La base de datos original de **XM** no es exclusiva del sistema Bogotá, sino de cobertura nacional. Sus características iniciales son:
* **Registros totales:** 63,991
* **Embalses monitoreados:** 26 embalses
* **Regiones hidrológicas:** 7 regiones
* **Frecuencia temporal:** Diaria
* **Ventana de tiempo:** 2019 – 2026

Dado que nuestro **Framework V7** está delimitado espacialmente a la **Cuenca del Río Bogotá**, se determinó la variable y el registro clave para la representatividad física del modelo:

**Filtro de Representatividad Espacial**
> `CodigoEmbalse == "AGREGADO_BOGOTA"`
>
> Este registro representa el volumen agregado del sistema de embalses que abastecen y regulan la cuenca de Bogotá (Chuza, Neusa, Sisga, Tominé, etc.). XM ya realiza la agregación física, lo que elimina el error de propagación por suma manual de componentes individuales y reduce el dataset a **~2,200 registros diarios**.

---

**2. Selección de Variables Físicas (Análisis de Relevancia)**

Contamos con cuatro variables numéricas originales en la base de XM. Evaluamos su potencial predictivo para los modelos LSTM:

| Variable | Tipo de Datos | Comportamiento Físico | Decisión en el Framework | Justificación Científica |
| :--- | :---: | :--- | :---: | :--- |
| **`CapacidadUtilMasa`** | Numérica | Prácticamente constante en el tiempo. | ❌ **Descartar** | Al ser la capacidad máxima de diseño, carece de variabilidad diaria/mensual. No aporta poder predictivo. |
| **`VolumenTotalMasa`** | Numérica | Incluye el volumen muerto (no operacional). | ❌ **Descartar** | El modelo LSTM requiere la dinámica del agua disponible para operación y regulación real. |
| **`VolumenUtilDiarioMasa`** | Numérica | Dinámica y directamente relacionada con la operación del embalse. | **⭐ Variable Principal** | Representa el agua útil disponible. Captura directamente la dinámica de sequías, regulación hídrica, abastecimiento y gobernanza del agua. |
| **`VertimientosMasa`** | Numérica | Sparsity alta (75% de los datos son 0). | ⚠️ **Variable Auxiliar** | Representa eventos extremos de rebose. No tiene comportamiento continuo, pero es de alto valor para análisis de resiliencia y riesgo. |

---

**3. Pipeline de Transformación de Datos (Data Prep Workflow)**

Para mantener la **coherencia temporal y dimensional** con el Framework V7 (el cual opera con frecuencia mensual), se ejecutará el siguiente flujo de procesamiento:

```text
[Dataset Crudo] 01_Capa_Reservas_Hidraulicas_Masa_V1.csv (~63.991 filas)
      │
      ▼
[Filtrar] CodigoEmbalse == "AGREGADO_BOGOTA" (~2.200 filas)
      │
      ▼
[Seleccionar] Fecha, VolumenUtilDiarioMasa, VertimientosMasa
      │
      ▼
[Cast / Parse] Convertir 'Fecha' a tipo DateTime
      │
      ▼
[Resample] Agrupar por Mes (Frecuencia Mensual 'MS')
      │
      ▼
[Agreggate] Calcular Promedio Mensual (mean)
      │
      ▼
[Output] Generar '01_Capa_Hydraulica_V1.xlsx' (~85 filas mensuales)

Conclusión

Considero que esta es una de las variables más valiosas del Framework. Mientras la capa ONI representa un forzante climático de gran escala y la capa hidrológica describe la respuesta del río, VolumenUtilDiarioMasa captura el estado operativo del sistema de embalses que regula el abastecimiento y la disponibilidad hídrica de la cuenca del Río Bogotá. Por ello, debería convertirse en la variable principal de la Capa Hidráulica, utilizando exclusivamente el registro AGREGADO_BOGOTA, consolidado a frecuencia mensual mediante el promedio de los valores diarios. Esta decisión mantiene la coherencia espacial y temporal del Framework V7 y evita incorporar información de embalses que no influyen directamente en el sistema hídrico objeto de estudio.